In [0]:
from pyspark.sql.types import StructType, DataType, StructField, StringType, IntegerType, DoubleType, TimestampType , FloatType
import pyspark.sql.functions as F


In [0]:
catalog = "ecommerce"

## defining schema for the data

brands_schema = StructType(
    [
        StructField("brand_code",StringType(),False),
        StructField("brand_name",StringType(),True),
        StructField("category_code",StringType(),True)
    ]
)


In [0]:
raw_brands_path = '/Volumes/ecommerce/source_data/raw/ecomm-raw-data/brands/'
df = spark.read.option('header','true') \
    .option('delimiter',',') \
    .schema(brands_schema) \
    .csv(raw_brands_path)

#adding metadata
brands_df = df.withColumn('_source_file',F.col('_metadata.file_path')) \
    .withColumn("injestedAt",F.current_timestamp())

display(df.limit(5))


In [0]:
brands_df.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema','True')\
    .saveAsTable(f'{catalog}.bronze.brz_brands')

In [0]:


date_schema = StructType([
    StructField("date",StringType(),True),
    StructField("year",IntegerType(),True),
    StructField("day_name",StringType(),True),
    StructField("quarter",IntegerType(),True),
    StructField("week_of_year",IntegerType(),True)
])

raw_date_path = '/Volumes/ecommerce/source_data/raw/ecomm-raw-data/date/'
date_df = spark.read.option('header',"true").option('delimiter',',').schema(date_schema).csv(raw_date_path)

#adding metadata
date_df = date_df.withColumn('_source_file',F.col('_metadata.file_path')) \
    .withColumn("injestedAt",F.current_timestamp())

display(date_df.limit(5))

date_df.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema','true')\
    .saveAsTable(f'{catalog}.bronze.brz_date')


In [0]:

category_schema = StructType([
    StructField('category_code',StringType(),False),
    StructField('category_name',StringType(),True),
])

raw_category_path = '/Volumes/ecommerce/source_data/raw/ecomm-raw-data/category/'

category_df = spark.read.option('header','true').option('delimiter',',').schema(category_schema).csv(raw_category_path)

category_df = category_df.withColumn('_source_file',F.col('_metadata.file_path'))\
    .withColumn('injestedAt',F.current_timestamp())

category_df.write.format('delta')\
    .mode('overwrite')\
    .options(mergeSchema='true')\
    .saveAsTable(f'{catalog}.bronze.brz_category')
display(category_df.limit(5))

In [0]:
customers_schema = StructType([
    StructField("customer_id",StringType(),False),
    StructField("phone",StringType(),True),
    StructField("country_code",StringType(),True),
    StructField("country",StringType(),True),
    StructField("state",StringType(),True)
])

raw_customers_path = '/Volumes/ecommerce/source_data/raw/ecomm-raw-data/customers/'
customers_df = spark.read.option('header',"true").option('delimiter',',').schema(customers_schema).csv(raw_customers_path)

customers_df = customers_df.withColumn('_source_file',F.col('_metadata.file_path')) \
    .withColumn("injestedAt",F.current_timestamp())

display(customers_df.limit(5))

customers_df.write.format('delta')\
    .mode('overwrite')\
    .option("mergeSchema", "true")\
    .saveAsTable(f'{catalog}.bronze.brz_customers')

In [0]:
#dt	order_ts	customer_id	order_id	item_seq	product_id	quantity	unit_price_currency	unit_price	discount_pct	tax_amount	channel	coupon_code

#order_items

order_items_schema = StructType([
    StructField("date",StringType(),True),
    StructField("order_ts",StringType(),True),
    StructField("customer_id",StringType(),True),
    StructField("order_id",IntegerType(),True),
    StructField("item_seq",IntegerType(),True),
    StructField("product_id",StringType(),False),
    StructField("quantity",StringType(),True),
    StructField("unit_price_currency",StringType(),True),
    StructField("unit_price",StringType(),True),
    StructField("discount_pct",StringType(),True),
    StructField("tax_amount",IntegerType(),False),
    StructField("channel",StringType(),True),
    StructField("coupon_code",StringType(),True),
])

raw_order_items_path = '/Volumes/ecommerce/source_data/raw/ecomm-raw-data/order_items/landing/'
order_items_df = spark.read.option('header',"true").option('delimiter',',').schema(order_items_schema).csv(raw_order_items_path)

order_items_df = order_items_df.withColumn('_source_file',F.col('_metadata.file_path')) \
    .withColumn("injestedAt",F.current_timestamp())

display(order_items_df.limit(5))

order_items_df.write.format('delta')\
    .mode('overwrite')\
    .option("mergeSchema", "true")\
    .saveAsTable(f'{catalog}.bronze.brz_order_items')

In [0]:
#product_id	sku	category_code	brand_code	color	size	material	weight_grams	length_cm	width_cm	height_cm	rating_count
#products


products_schema = StructType([
    StructField("product_id",StringType(),True),
    StructField("sku",StringType(),True),
    StructField("category_code",StringType(),True),
    StructField("brand_code",StringType(),True),
    StructField("color",StringType(),True),
    StructField("size",StringType(),False),
    StructField("material",StringType(),True),
    StructField("weight_grams",StringType(),True),
    StructField("length_cm",StringType(),True),
    StructField("width_cm",DoubleType(),True),
    StructField("height_cm",DoubleType(),False),
    StructField("rating_count",IntegerType(),True)
])

raw_products_path = '/Volumes/ecommerce/source_data/raw/ecomm-raw-data/products/'
products_df = spark.read.option('header',"true").option('delimiter',',').schema(products_schema).csv(raw_products_path)

products_df = products_df.withColumn('_source_file',F.col('_metadata.file_path')) \
    .withColumn("injestedAt",F.current_timestamp())

display(products_df.limit(5))

products_df.write.format('delta')\
    .mode('overwrite')\
    .option("overwriteSchema", "true")\
    .saveAsTable(f'{catalog}.bronze.brz_products')
